In [3]:
pip install numpy

   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.6 MB 653.6 kB/s eta 0:00:20
   ---------------------------------------- 0.1/12.6 MB 1.4 MB/s eta 0:00:09
    --------------------------------------- 0.2/12.6 MB 1.8 MB/s eta 0:00:07
   - -------------------------------------- 0.4/12.6 MB 2.1 MB/s eta 0:00:06
   - -------------------------------------- 0.4/12.6 MB 2.0 MB/s eta 0:00:07
   - -------------------------------------- 0.5/12.6 MB 2.0 MB/s eta 0:00:07
   -- ------------------------------------- 0.6/12.6 MB 2.1 MB/s eta 0:00:06
   -- ------------------------------------- 0.7/12.6 MB 2.2 MB/s eta 0:00:06
   -- ------------------------------------- 0.9/12.6 MB 2.3 MB/s eta 0:00:06
   --- ------------------------------------ 1.0/12.6 MB 2.3 MB/s eta 0:00:05
   --- ------------------------------------ 1.1/12.6 MB 2.4 MB/s eta 0:00:05
   --- ----


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# LIF Neuron (Phase 1)

This notebook defines a simple Leaky Integrate-and-Fire (LIF) neuron class for Phase 1 of the SNN implementation roadmap.

In [4]:
"""Simple Leaky Integrate-and-Fire (LIF) neuron implementation (Phase 1)

Usage: import `LIFNeuron` and call `simulate()` or run `runner.ipynb`.
"""
import numpy as np

class LIFNeuron:
    def __init__(self, tau_m=20e-3, R=100e6, v_rest=-65e-3, v_reset=-65e-3, v_th=-50e-3, dt=1e-4):
        """Create a single LIF neuron.

        Args:
            tau_m: membrane time constant (s)
            R: membrane resistance (Ohm)
            v_rest: resting potential (V)
            v_reset: reset potential after spike (V)
            v_th: spike threshold (V)
            dt: simulation timestep (s)
        """
        self.tau_m = float(tau_m)
        self.R = float(R)
        self.v_rest = float(v_rest)
        self.v_reset = float(v_reset)
        self.v_th = float(v_th)
        self.dt = float(dt)
        self.v = self.v_rest

    def reset(self):
        """Reset membrane potential to resting value."""
        self.v = self.v_rest

    def step(self, I_ext):
        """Advance neuron by one timestep given external current (A).

        Returns:
            spike (int): 1 if spike occurred at this step, else 0
        """
        dv = (-(self.v - self.v_rest) + self.R * I_ext) * (self.dt / self.tau_m)
        self.v += dv
        if self.v >= self.v_th:
            # emit spike and reset membrane potential
            self.v = self.v_reset
            return 1
        return 0

    def simulate(self, I, dt=None):
        """Simulate the neuron over a current time series.

        Args:
            I: array-like external current (A) with length N
            dt: optional timestep (overrides self.dt)

        Returns:
            spikes: numpy array of shape (N,) with 0/1 spike events
            V: numpy array of membrane potentials (V)
        """
        if dt is None:
            dt = self.dt
        I = np.asarray(I, dtype=float)
        n = I.shape[0]
        spikes = np.zeros(n, dtype=int)
        V = np.zeros(n, dtype=float)
        for i in range(n):
            spikes[i] = self.step(I[i])
            V[i] = self.v
        return spikes, V